In [61]:
import pandas as pd
import numpy as np

In [62]:
df = pd.read_csv(r'C:\Users\Junayed\pandas_prac\Aug_5\messy_bank_day5.csv')

In [63]:
df.head(5)

,TransactionID,Account Holder,email,TransactionDate,Merchant,Category Tag,Amount,type,branch_code,PaymentMethod,Is_International,Status,notes
0,T1001,Owen Marsh,owen.marsh@gmail.com,2023-06-01,Whole Foods,Groceries/Food,-84.20,Debit,452,Debit Card,No,Completed,NaN
1,T1002,Bianca Ruiz,bianca.ruiz@gmail.com,2023-06-01,Amazon,Shopping > Online,-142.99,debit,452,Credit Card,No,completed,NaN
2,T1003,Femi Adeyemi,femi.adeyemi@gmail.com,2023-06-02,Employer Inc,Income-Salary,3200.00,Credit,478,Direct Deposit,No,COMPLETED,NaN
3,T1004,Katarina Novak,katarina.novak@hotmail.com,2023-06-02,Netflix,Entertainment/Streaming,(15.99),Debit,452,Debit Card,No,Completed,
4,T1005,Owen Marsh,owen.marsh@gmail.com,2023-06-01,Whole Foods,Groceries/Food,-84.20,Debit,452,Debit Card,No,Completed,NaN


In [100]:
df.shape

(32, 14)

In [101]:
df.dtypes

transactionid           str
account_holder          str
email                   str
transactiondate         str
merchant                str
main_category        object
sub_category         object
amount              float64
type                    str
branch_code           int64
paymentmethod           str
is_international        str
status                  str
notes                   str
dtype: object

In [64]:
df.columns.tolist()

[' TransactionID',
 'Account Holder',
 'email ',
 'TransactionDate',
 'Merchant',
 'Category Tag',
 'Amount',
 'type',
 'branch_code',
 'PaymentMethod',
 'Is_International',
 'Status',
 'notes']

## Column Name Normalization:

**Issue Identified:**  
After inspecting the dataset columns using `df.columns.tolist()`, several formatting inconsistencies were found:
* Leading and trailing whitespace
* Irregular spacing between words
* Inconsistent casing (mixed UPPERCASE and lowercase)
* Missing underscores as word separators

**Fix Applied:**  
Standardized all column names to **snake_case** by:
1. Stripping leading and trailing spaces.
2. Converting all characters to lowercase.
3. Replacing spaces with underscores (`_`).

In [65]:
def clean_columns_name(df):
    df.columns = (
        df.columns.str.strip().str.lower().str.replace(" ", "_", regex = True)
    )
    return df

# running the function.

df = clean_columns_name(df)
df.columns.tolist()

['transactionid',
 'account_holder',
 'email',
 'transactiondate',
 'merchant',
 'category_tag',
 'amount',
 'type',
 'branch_code',
 'paymentmethod',
 'is_international',
 'status',
 'notes']

In [66]:
df["transactiondate"] = pd.to_datetime(df["transactiondate"], format = 'mixed').dt.strftime("%Y-%m-%d")
df["transactiondate"]

0     2023-06-01
1     2023-06-01
2     2023-06-02
3     2023-06-02
4     2023-06-01
5     2023-06-03
6     2023-06-03
7     2023-06-04
8     2023-06-04
9     2023-06-03
10    2023-06-05
11    2023-06-05
12    2023-06-06
13    2023-06-06
14    2023-06-07
15    2023-06-07
16    2023-06-08
17    2023-06-08
18    2023-06-09
19    2023-06-09
20    2023-06-10
21    2023-06-10
22    2023-06-11
23    2023-06-11
24    2023-06-01
25    2023-06-12
26    2023-06-12
27    2023-06-13
28    2023-06-13
29    2023-06-14
30    2023-06-14
31    2023-06-15
32    2023-06-15
33    2023-06-16
34    2023-06-16
Name: transactiondate, dtype: str

> Cleaning `Merchant` column.

In [67]:
df["merchant"] = df["merchant"].str.strip().str.title()
df["merchant"]

0         Whole Foods
1              Amazon
2        Employer Inc
3             Netflix
4         Whole Foods
5           Shell Gas
6               Amazn
7        Employer Inc
8      Delta Airlines
9              Amazon
10          Starbucks
11               Uber
12       Employer Inc
13           Best Buy
14       Cvs Pharmacy
15            Netflix
16         Air France
17        Trader Joes
18       Employer Inc
19               Lyft
20          Walgreens
21             Amazon
22       Employer Inc
23    United Airlines
24             Amazon
25        Whole Foods
26            Spotify
27          Shell Gas
28       Employer Inc
29       Cvs Pharmacy
30               Uber
31              Amazn
32       Employer Inc
33        Trader Joes
34     Delta Airlines
Name: merchant, dtype: str

In [68]:
df['category_tag']

0              Groceries/Food
1           Shopping > Online
2               Income-Salary
3     Entertainment/Streaming
4              Groceries/Food
5                   Auto-Fuel
6             Shopping/Online
7             Income > Salary
8              Travel-Flights
9             Shopping/Online
10           Groceries > Food
11        Transport-Rideshare
12              Income/Salary
13       Shopping-Electronics
14            Health/Pharmacy
15    Entertainment-Streaming
16             Travel/Flights
17             Groceries-Food
18              Income-Salary
19        Transport/Rideshare
20            Health-Pharmacy
21            Shopping/Online
22            Income > Salary
23             Travel-Flights
24            Shopping/Online
25           Groceries > Food
26    Entertainment/Streaming
27                  Auto/Fuel
28              Income-Salary
29            Health-Pharmacy
30        Transport-Rideshare
31            Shopping-Online
32              Income/Salary
33        

## Category Parsing: Normalization & Hierarchy Creation

**Issue Identified:**  
 horizontal separators (`/`, `>`, `-`) are inconsistently used to separate main categories from subcategories in the `category_tag` column.

**Fix Applied:**  
1. **Standardized Separators:** Replaced all variations (`/`, `>`, `-`) with a white space (` `) and removed extra spaces.
2. **Category Hierarchy Splitting:** Split the cleaned tag into two distinct columns:
   * **`Main Category`** (`split[0]`): The primary level category.
   * **`Subcategory`** (`split[1]`): The specific secondary classification.

In [69]:
df["category_tag"] = df["category_tag"].str.replace(">", " ").str.replace("/", " ").str.replace("-", " ")
# Spliting 
split = df["category_tag"].str.split()
# making new columns.
df["main_category"] = split.str[0]
df["sub_category"] = split.str[1]
df["main_category"] = df["main_category"].str.strip()
df["sub_category"] = df["sub_category"].str.strip()
df[["category_tag", "main_category", "sub_category"]]

,category_tag,main_category,sub_category
0,Groceries Food,Groceries,Food
1,Shopping Online,Shopping,Online
2,Income Salary,Income,Salary
3,Entertainment Streaming,Entertainment,Streaming
4,Groceries Food,Groceries,Food
5,Auto Fuel,Auto,Fuel
6,Shopping Online,Shopping,Online
7,Income Salary,Income,Salary
8,Travel Flights,Travel,Flights
9,Shopping Online,Shopping,Online


## Column Reordering & Cleanup:

**Fix Applied:**  
1. **Repositioned New Columns:** Inserted `Main Category` and `Subcategory` immediately after the original `category_tag` column for better visual alignment and logical flow.
2. **Removed Redundant Column:** Dropped the original `category_tag` column now that its information has been successfully extracted into structured categories.

In [70]:
target_col = df.columns.get_loc("category_tag")

df.insert(target_col +1, "main_category", df.pop("main_category"))
df.insert(target_col +2, "sub_category", df.pop("sub_category"))
df.head(5)

,transactionid,account_holder,email,transactiondate,merchant,category_tag,main_category,sub_category,amount,type,branch_code,paymentmethod,is_international,status,notes
0,T1001,Owen Marsh,owen.marsh@gmail.com,2023-06-01,Whole Foods,Groceries Food,Groceries,Food,-84.20,Debit,452,Debit Card,No,Completed,NaN
1,T1002,Bianca Ruiz,bianca.ruiz@gmail.com,2023-06-01,Amazon,Shopping Online,Shopping,Online,-142.99,debit,452,Credit Card,No,completed,NaN
2,T1003,Femi Adeyemi,femi.adeyemi@gmail.com,2023-06-02,Employer Inc,Income Salary,Income,Salary,3200.00,Credit,478,Direct Deposit,No,COMPLETED,NaN
3,T1004,Katarina Novak,katarina.novak@hotmail.com,2023-06-02,Netflix,Entertainment Streaming,Entertainment,Streaming,(15.99),Debit,452,Debit Card,No,Completed,
4,T1005,Owen Marsh,owen.marsh@gmail.com,2023-06-01,Whole Foods,Groceries Food,Groceries,Food,-84.20,Debit,452,Debit Card,No,Completed,NaN


In [72]:
df = df.drop(columns="category_tag")
df.head(5)


,transactionid,account_holder,email,transactiondate,merchant,main_category,sub_category,amount,type,branch_code,paymentmethod,is_international,status,notes
0,T1001,Owen Marsh,owen.marsh@gmail.com,2023-06-01,Whole Foods,Groceries,Food,-84.20,Debit,452,Debit Card,No,Completed,NaN
1,T1002,Bianca Ruiz,bianca.ruiz@gmail.com,2023-06-01,Amazon,Shopping,Online,-142.99,debit,452,Credit Card,No,completed,NaN
2,T1003,Femi Adeyemi,femi.adeyemi@gmail.com,2023-06-02,Employer Inc,Income,Salary,3200.00,Credit,478,Direct Deposit,No,COMPLETED,NaN
3,T1004,Katarina Novak,katarina.novak@hotmail.com,2023-06-02,Netflix,Entertainment,Streaming,(15.99),Debit,452,Debit Card,No,Completed,
4,T1005,Owen Marsh,owen.marsh@gmail.com,2023-06-01,Whole Foods,Groceries,Food,-84.20,Debit,452,Debit Card,No,Completed,NaN


In [73]:
df["amount"]

0      -84.20
1     -142.99
2     3200.00
3     (15.99)
4      -84.20
5      -52.10
6      -38.40
7     2900.00
8     -410.00
9      -38.40
10      -6.75
11    (24.50)
12    3100.00
13    -299.99
14     -18.60
15     -15.99
16    -560.00
17     -63.15
18    2950.00
19     -19.30
20     -27.85
21     -91.20
22    3050.00
23    -320.00
24    -142.99
25     -71.40
26      -9.99
27     -48.00
28    3200.00
29     -22.10
30     -31.75
31     -54.60
32    2980.00
33     -58.30
34    -275.00
Name: amount, dtype: str

## Amount Normalization

**Issue Identified:**  
The `Amount` column contains raw text values with potential non-numeric noise or formatting artifacts that need to be cleaned while maintaining debit indicator signs.

**Fix Applied:**  
1. **Sanitized String Values:** Used regex `[^\d.-]` to strip out all non-numeric characters, preserving only digits, decimal points, and negative signs (`-`).
2. **Preserved Transaction Types:** Retained negative signs (`-`) to represent **debit amounts** accurately.
3. **Type Conversion:** Converted the cleaned string column into numeric (`float`) values for downstream calculations.

In [75]:
import re
def clean_amount(val):
    val = str(val).strip()
    neg = val.startswith("(") and val.endswith(")")
    num_str = re.sub(r"[^\d.\-]", "", val)
    result = float(num_str)
    return -abs(result) if neg else result

df["amount"] = df["amount"].apply(clean_amount)
df["amount"]

0      -84.20
1     -142.99
2     3200.00
3      -15.99
4      -84.20
5      -52.10
6      -38.40
7     2900.00
8     -410.00
9      -38.40
10      -6.75
11     -24.50
12    3100.00
13    -299.99
14     -18.60
15     -15.99
16    -560.00
17     -63.15
18    2950.00
19     -19.30
20     -27.85
21     -91.20
22    3050.00
23    -320.00
24    -142.99
25     -71.40
26      -9.99
27     -48.00
28    3200.00
29     -22.10
30     -31.75
31     -54.60
32    2980.00
33     -58.30
34    -275.00
Name: amount, dtype: float64

## `type` vs `amount` — cross-field 

**The rule:** a `Debit` should be a negative amount (money leaving the account), a `Credit` should be positive (money coming in). 

In [77]:
df["type"] = df["type"].str.strip().str.title()


In [79]:
mismatch = ((df["type"] == "Debit") & (df["amount"] > 0)) | ((df["type"] == "Credit") & (df["amount"] < 0))
df.loc[mismatch, ["transactionid", "merchant", "type", "amount", "notes"]]

,transactionid,merchant,type,amount,notes
15,T1016,Netflix,Credit,-15.99,Sign mismatch?


**Only one mismatch we have found at T1016, Credit should be positive but the amount is negative which indicates that the type `Debit`. If we look at T1015 it has the note `Sign mismatch?` which is not true as `Debit` amount always will be negative.**

**Fix:** Update the amount to positive at T1016

In [81]:
df.loc[df["transactionid"] == "T1016", "type"] = "Debit"

In [83]:
df["status"] = df["status"].str.strip().str.title()
df["status"]

0     Completed
1     Completed
2     Completed
3     Completed
4     Completed
5     Completed
6     Completed
7     Completed
8     Completed
9     Completed
10    Completed
11    Completed
12      Pending
13    Completed
14    Completed
15    Completed
16    Completed
17    Completed
18       Failed
19    Completed
20    Completed
21    Completed
22    Completed
23    Completed
24    Completed
25    Completed
26    Completed
27    Completed
28    Completed
29    Completed
30    Completed
31    Completed
32    Completed
33    Completed
34    Completed
Name: status, dtype: str

In [85]:
df["notes"].replace(("","na","nan"), np.nan)

0                     NaN
1                     NaN
2                     NaN
3                        
4                     NaN
5                     NaN
6     Merchant name typo?
7                     NaN
8                     NaN
9              Duplicate?
10                    NaN
11                    NaN
12                    NaN
13                    NaN
14         Sign mismatch?
15         Sign mismatch?
16                    NaN
17                    NaN
18                    NaN
19                    NaN
20                    NaN
21                    NaN
22                    NaN
23                    NaN
24             Duplicate?
25                    NaN
26                    NaN
27                    NaN
28                    NaN
29                    NaN
30                    NaN
31    Merchant name typo?
32                    NaN
33                    NaN
34                    NaN
Name: notes, dtype: str

- At T004 the np.nan couldn't replace it to NaN, why? let's use repr to check.

In [86]:
df['notes'].apply(repr)

0                       nan
1                       nan
2                       nan
3                    '\xa0'
4                       nan
5                       nan
6     'Merchant name typo?'
7                       nan
8                       nan
9              'Duplicate?'
10                      nan
11                      nan
12                      nan
13                      nan
14         'Sign mismatch?'
15         'Sign mismatch?'
16                      nan
17                      nan
18                      nan
19                      nan
20                      nan
21                      nan
22                      nan
23                      nan
24             'Duplicate?'
25                      nan
26                      nan
27                      nan
28                      nan
29                      nan
30                      nan
31    'Merchant name typo?'
32                      nan
33                      nan
34                      nan
Name: notes, dtype: 

- We see a raw value '\xa0' - a non-breaking space. Let's replace this now with " " and then again we will use np.nan.

In [87]:
df["notes"] = df["notes"].astype(str).str.replace("\xa0", " ", regex=True).str.strip()
df["notes"] = df["notes"].replace(("","na","nan"), np.nan)
df["notes"]

0                     NaN
1                     NaN
2                     NaN
3                     NaN
4                     NaN
5                     NaN
6     Merchant name typo?
7                     NaN
8                     NaN
9              Duplicate?
10                    NaN
11                    NaN
12                    NaN
13                    NaN
14         Sign mismatch?
15         Sign mismatch?
16                    NaN
17                    NaN
18                    NaN
19                    NaN
20                    NaN
21                    NaN
22                    NaN
23                    NaN
24             Duplicate?
25                    NaN
26                    NaN
27                    NaN
28                    NaN
29                    NaN
30                    NaN
31    Merchant name typo?
32                    NaN
33                    NaN
34                    NaN
Name: notes, dtype: str

In [90]:
df.loc[df["merchant"] == "Amazn", "merchant"] = "Amazon"

df["merchant"].value_counts()

merchant
Employer Inc       7
Amazon             6
Whole Foods        3
Netflix            2
Shell Gas          2
Delta Airlines     2
Uber               2
Cvs Pharmacy       2
Trader Joes        2
Starbucks          1
Best Buy           1
Air France         1
Lyft               1
Walgreens          1
United Airlines    1
Spotify            1
Name: count, dtype: int64

# Duplicates

In [92]:
dup_cols = ["account_holder", "email", "merchant", "transactiondate", "amount"]

df[df.duplicated(subset=dup_cols, keep=False)]

,transactionid,account_holder,email,transactiondate,merchant,main_category,sub_category,amount,type,branch_code,paymentmethod,is_international,status,notes
0,T1001,Owen Marsh,owen.marsh@gmail.com,2023-06-01,Whole Foods,Groceries,Food,-84.20,Debit,452,Debit Card,No,Completed,NaN
1,T1002,Bianca Ruiz,bianca.ruiz@gmail.com,2023-06-01,Amazon,Shopping,Online,-142.99,Debit,452,Credit Card,No,Completed,NaN
4,T1005,Owen Marsh,owen.marsh@gmail.com,2023-06-01,Whole Foods,Groceries,Food,-84.20,Debit,452,Debit Card,No,Completed,NaN
6,T1007,Priya Kapoor,priya.kapoor@gmail.com,2023-06-03,Amazon,Shopping,Online,-38.40,Debit,452,Credit Card,No,Completed,Merchant name typo?
9,T1010,Priya Kapoor,priya.kapoor@gmail.com,2023-06-03,Amazon,Shopping,Online,-38.40,Debit,452,Credit Card,No,Completed,Duplicate?
24,T1025,Bianca Ruiz,bianca.ruiz@gmail.com,2023-06-01,Amazon,Shopping,Online,-142.99,Debit,452,Credit Card,No,Completed,Duplicate?


In [94]:
df = df.drop_duplicates(subset=dup_cols, keep='first').reset_index(drop=True)

In [97]:
df.shape

(32, 14)

In [98]:
df.dtypes

transactionid           str
account_holder          str
email                   str
transactiondate         str
merchant                str
main_category        object
sub_category         object
amount              float64
type                    str
branch_code           int64
paymentmethod           str
is_international        str
status                  str
notes                   str
dtype: object

In [99]:
df.to_csv("Cleaned_bank_data.csv", index=False)

df.to_excel("Cleaned_bank_data.xlsx", index=False)